# Entity Framework & DDD

In [ ]:
class Product
{
    public readonly record struct ProductId(Guid Value) // DDD purist dont like; just a Guid; they prefer
    {                                                   // explicit identity, invariants, and type safety
        public static ProductId New() => new(Guid.NewGuid());
    }

    public ProductId Id { get; private set; } = ProductId.New();
    public string Name { get; private set; }
    public decimal UnitPrice { get; private set; }

    public Product(string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(name))
            throw new ArgumentException("Name required.");

        if (unitPrice < 0)
            throw new ArgumentOutOfRangeException(nameof(unitPrice));

        Name = name;
        UnitPrice = unitPrice;
    }

    public void Rename(string name)
    {
        if (string.IsNullOrWhiteSpace(name))
            throw new ArgumentException("Name required.");

        Name = name;
    }

    public void ChangePrice(decimal price)
    {
        if (price < 0)
            throw new ArgumentOutOfRangeException(nameof(price));

        UnitPrice = price;
    }
}

class ReceiptItem
{
    public ReceiptItem(Product.ProductId productId, string productName, decimal unitPrice, int quantity)
    {
        if (quantity <= 0)
            throw new ArgumentOutOfRangeException(nameof(quantity));

        (this.ProductId, this.ProductName, this.UnitPrice, this.Quantity) =
            (productId, productName, unitPrice, quantity);
    }

    public Product.ProductId ProductId { get; private set; }
    public string ProductName { get; private set; }
    public decimal UnitPrice { get; private set; }
    public int Quantity { get; private set; }

    public decimal LineTotal => UnitPrice * Quantity;
}

class Receipt
{
    public readonly record struct ReceiptId(Guid Value)
    {
        public static ReceiptId New() => new(Guid.NewGuid());
    }

    readonly List<ReceiptItem> items = new();

    public ReceiptId Id { get; private set; } = ReceiptId.New();
    public DateTime IssuedAt { get; private set; } = DateTime.UtcNow;
    public IReadOnlyCollection<ReceiptItem> Items => items.AsReadOnly();
    public decimal Total => items.Sum(x => x.LineTotal);

    public void AddItem(Product product, int quantity)
    {
        ArgumentNullException.ThrowIfNull(product);

        items.Add(new ReceiptItem(product.Id, product.Name, product.UnitPrice, quantity));
    }

    public void RemoveItem(Product.ProductId productId)
        => items.RemoveAll(x => x.ProductId == productId);
}

In [ ]:
#r "nuget: Microsoft.EntityFrameworkCore"

In [ ]:
using Microsoft.EntityFrameworkCore.Storage.ValueConversion;
using System.Linq.Expressions;

abstract class StronglyTypedIdConverter<TId> : ValueConverter<TId, Guid> where TId : struct
{
    protected StronglyTypedIdConverter(Expression<Func<TId, Guid>> toProvider, Expression<Func<Guid, TId>> fromProvider)
    : base(toProvider, fromProvider)
    { }
}

sealed class ProductIdConverter : StronglyTypedIdConverter<Product.ProductId>
{
    public ProductIdConverter() : base(
        id => id.Value,
        value => new Product.ProductId(value))
    { }
}

sealed class ReceiptIdConverter : StronglyTypedIdConverter<Receipt.ReceiptId>
{
    public ReceiptIdConverter() : base(
        id => id.Value,
        value => new Receipt.ReceiptId(value))
    { }
}

In [ ]:
using Microsoft.EntityFrameworkCore;
using Microsoft.EntityFrameworkCore.Metadata.Builders;

sealed class ProductConfiguration : IEntityTypeConfiguration<Product>
{
    public void Configure(EntityTypeBuilder<Product> builder)
    {
        //builder.ToTable("Products");
        builder.Property<int>("Id")
            .ValueGeneratedOnAdd();
        builder.HasKey("Id");
        builder.Property(x => x.Id)
            .HasConversion(new ProductIdConverter())
            .IsRequired();
        builder.HasAlternateKey(x => x.Id);

        builder.Property(x => x.Name)
            .IsRequired()
            .HasMaxLength(200);

        builder.Property(x => x.UnitPrice)
            .HasPrecision(18, 2);
    }
}

sealed class ReceiptItemConfiguration : IEntityTypeConfiguration<ReceiptItem>
{
    public void Configure(EntityTypeBuilder<ReceiptItem> builder)
    {
        //builder.ToTable("ReceiptItems");
        builder.Property<int>("Id")
            .ValueGeneratedOnAdd();
        builder.HasKey("Id");
        builder.Property(x => x.ProductId)
            .HasConversion(new ProductIdConverter())
            .IsRequired();

        builder.Property(x => x.ProductName)
            .IsRequired()
            .HasMaxLength(200);

        builder.Property(x => x.UnitPrice)
            .HasPrecision(18, 2);

        builder.Property(x => x.Quantity)
            .IsRequired();

        builder.Ignore(x => x.LineTotal);
    }
}

sealed class ReceiptConfiguration : IEntityTypeConfiguration<Receipt>
{
    public void Configure(EntityTypeBuilder<Receipt> builder)
    {
        // builder.ToTable("Receipts");

        builder.Property<int>("Id")
            .ValueGeneratedOnAdd();
        builder.HasKey("Id");
        builder.Property(x => x.Id)
            .HasConversion(new ReceiptIdConverter())
            .IsRequired();
        builder.HasAlternateKey(x => x.Id);

        builder.Property(x => x.IssuedAt)
            .IsRequired();

        builder.Ignore(x => x.Total);

        builder.HasMany<ReceiptItem>("items")
            .WithOne()
            .HasForeignKey("ReceiptId")
            .OnDelete(DeleteBehavior.Cascade);

        builder.Navigation("items")
            .UsePropertyAccessMode(PropertyAccessMode.Field);
    }
}


In [ ]:
using Microsoft.EntityFrameworkCore;

sealed class PosDbContext : DbContext
{
    public PosDbContext(DbContextOptions<PosDbContext> options) : base(options)
    { }

    public DbSet<Product> Products => Set<Product>();
    public DbSet<Receipt> Receipts => Set<Receipt>();

    protected override void OnModelCreating(ModelBuilder modelBuilder)
    {
        modelBuilder.ApplyConfigurationsFromAssembly(typeof(PosDbContext).Assembly);

        base.OnModelCreating(modelBuilder);
    }
}

In [ ]:
class ActionResult {}
class NotFound : ActionResult {}
class BadRequest(string Reason) : ActionResult {}
class Ok(object Result) : ActionResult {}

record AddReceiptItemRequest(Guid ProductId, int Quantity);

class ReceiptsController // Controller
{
    readonly PosDbContext db;

    public ReceiptsController(PosDbContext db) => (this.db) = db;

    public async Task<ActionResult> AddItem(Guid receiptId, AddReceiptItemRequest request)
    {
        var receipt = await db.Receipts.FirstOrDefaultAsync(r => r.Id == new Receipt.ReceiptId(receiptId));
        if (receipt is null) return new NotFound();

        var productId = new Product.ProductId(request.ProductId);
        var product = await db.Products.FirstOrDefaultAsync(p => p.Id == productId);
        if (product is null) return new BadRequest("Product not found.");

        receipt.AddItem(product, request.Quantity);
        await db.SaveChangesAsync();

        return new Ok(new
        {
            receiptId,
            total = receipt.Total
        });
    }
}